In [3]:
#1
import pymysql
from pymysql.cursors import DictCursor

class CafeDBManager:
    def __init__(self, host, user, password, db, port=3306):
        self.db_config = {
            'host': host,
            'user': user,
            'password': password,
            'database': db,
            'port': port,
            'charset': 'utf8mb4', 
            'cursorclass': DictCursor 
        }

    def execute_query(self, sql, params=None):
        conn = pymysql.connect(**self.db_config)
        
        try:
            with conn.cursor() as cur:
                cur.execute(sql, params)
                result = cur.fetchall()
                return result
        except Exception as e:
            print(f"[조회 에러] {e}")
            raise e
        finally:
            conn.close() 

    def execute_update(self, sql, params=None):
        conn = pymysql.connect(**self.db_config)
        
        try:
            with conn.cursor() as cur:
                cur.execute(sql, params) 
           
            conn.commit() 
            print("업데이트 성공")
            
        except Exception as e:
            conn.rollback() 
            print(f"[업데이트 에러 발생] 작업이 롤백되었습니다: {e}")
        finally:
            conn.close() 

if __name__ == "__main__":
    cafe_db = CafeDBManager(
        host='127.0.0.1',
        user='root',
        password='test1234', 
        db='cafe_db'
    )
    
    select_sql = "SELECT * FROM tb_menu WHERE category_id = %s"
    menus = cafe_db.execute_query(select_sql, (1,)) 
    print(menus) 

    update_sql = "UPDATE tb_menu SET price = %s WHERE menu_nm = %s"
    cafe_db.execute_update(update_sql, (4500, '카페라떼'))

[{'menu_id': 1, 'category_id': 1, 'menu_nm': '아메리카노', 'price': Decimal('4500.00'), 'is_seasonal': 0}, {'menu_id': 2, 'category_id': 1, 'menu_nm': '카페라떼', 'price': Decimal('5000.00'), 'is_seasonal': 0}]
업데이트 성공


In [15]:
import pymysql
import pandas as pd
from pymysql.cursors import DictCursor

class CafeDBManager:
    def __init__(self, host, user, password, db, port=3306):
        self.db_config = {
            'host': host, 'user': user, 'password': password,
            'database': db, 'port': port,
            'charset': 'utf8mb4', 'cursorclass': DictCursor
        }

    def execute_query(self, sql, params=None):
        conn = pymysql.connect(**self.db_config)
        try:
            with conn.cursor() as cur:
                cur.execute(sql, params)
                return cur.fetchall()
        finally:
            conn.close()

if __name__ == "__main__":
    db = CafeDBManager(host='127.0.0.1', user='root', password='test1234', db='cafe_db')

    print("[분석 1] 매장별 매출 및 객단가(AOV) 진단\n")
    
    sql_1 = """
        SELECT 
            s.store_nm,
            COUNT(DISTINCT o.order_id) AS order_cnt,
            SUM(i.qty * i.unit_price) AS total_sales
        FROM tb_store s
        JOIN tb_order o ON s.store_id = o.store_id
        JOIN tb_order_item i ON o.order_id = i.order_id
        GROUP BY s.store_nm
    """
    df1 = pd.DataFrame(db.execute_query(sql_1))
    
    if not df1.empty:
        df1['total_sales'] = df1['total_sales'].astype(float)
        df1['order_cnt'] = df1['order_cnt'].astype(float)
        
        df1['AOV'] = df1.apply(lambda row: round(row['total_sales'] / row['order_cnt'], 0) if row['order_cnt'] > 0 else 0, axis=1)
        df1 = df1.sort_values(by='total_sales', ascending=False)
        
        print(df1.to_string(index=False))
        
        top_store = df1.iloc[0]
        print(f"\n'{top_store['store_nm']}'이(가) 총 매출액 {int(top_store['total_sales']):,}원으로 가장 높은 성과를 달성했습니다.")
    else:
        print("데이터가 없습니다.")
    
    
    print("\n[분석 2] 시간대별 주문 피크 타임 분석\n")
    
    sql_2 = """
        SELECT 
            HOUR(order_dt) AS hour,
            COUNT(*) AS order_cnt
        FROM tb_order
        GROUP BY HOUR(order_dt)
    """
    df2 = pd.DataFrame(db.execute_query(sql_2))
    
    if not df2.empty:
        peak_times = df2.sort_values(by='order_cnt', ascending=False).head(3)
        
        print("TOP 3 피크 시간대:")
        for idx, row in peak_times.iterrows():
            print(f" - {int(row['hour'])}시: {row['order_cnt']}건")
        
        top_hour = int(peak_times.iloc[0]['hour'])
        print(f"\n가장 주문이 몰리는 {top_hour}시 전후로 매장 파트타임(알바) 인력을 집중적으로 배치하는것을 추천합니다.")
    else:
        print("데이터가 없습니다.")

    print("\n[분석 3] 메뉴 카테고리별 매출 점유율 분석\n")
    
    sql_3 = """
        SELECT 
            c.category_nm,
            IFNULL(SUM(i.qty * i.unit_price), 0) AS category_sales,
            IFNULL(SUM(i.qty), 0) AS total_qty
        FROM tb_menu_category c
        JOIN tb_menu m ON c.category_id = m.category_id
        LEFT JOIN tb_order_item i ON m.menu_id = i.menu_id
        GROUP BY c.category_nm
    """
    df3 = pd.DataFrame(db.execute_query(sql_3))
    
    if not df3.empty:
        df3['category_sales'] = df3['category_sales'].astype(float)
        total_all_sales = df3['category_sales'].sum()

        if total_all_sales > 0:
            df3['sales_share_percent'] = ((df3['category_sales'] / total_all_sales) * 100).round(1)
        else:
            df3['sales_share_percent'] = 0.0
            
        df3 = df3.sort_values(by='category_sales', ascending=False)
        
        print(df3.to_string(index=False))
        
        top_category = df3.iloc[0]
        print(f"\n[{top_category['category_nm']}] 카테고리가 전체 매출의 {top_category['sales_share_percent']}%를 견인하는 핵심 주력 제품군입니다.")
    else:
        print("데이터가 없습니다.")

[분석 1] 매장별 매출 및 객단가(AOV) 진단

store_nm  order_cnt  total_sales     AOV
    강남본점        3.0      27000.0  9000.0
     홍대점        1.0      13500.0 13500.0

'강남본점'이(가) 총 매출액 27,000원으로 가장 높은 성과를 달성했습니다.

[분석 2] 시간대별 주문 피크 타임 분석

TOP 3 피크 시간대:
 - 12시: 2건
 - 13시: 1건
 - 18시: 1건

가장 주문이 몰리는 12시 전후로 매장 파트타임(알바) 인력을 집중적으로 배치하는것을 추천합니다.

[분석 3] 메뉴 카테고리별 매출 점유율 분석

category_nm  category_sales total_qty  sales_share_percent
         커피         27500.0         6                 67.9
        디저트         13000.0         2                 32.1

[커피] 카테고리가 전체 매출의 67.9%를 견인하는 핵심 주력 제품군입니다.
